In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import pandas as pd

PROJECT_DIR = Path(
    "/content/drive/MyDrive/SleepStageClassificationProject"
)

RESULTS_DIR = PROJECT_DIR / "results"
DATASET_DIR = PROJECT_DIR / "dataset"

print("Project exists:", PROJECT_DIR.exists())
print("Results exists:", RESULTS_DIR.exists())
print("Dataset exists:", DATASET_DIR.exists())

In [ ]:
def scan_directory(base_dir):

    rows = []

    for root, dirs, files in os.walk(base_dir):
        dirs.sort()
        files.sort()

        for filename in files:
            path = Path(root) / filename

            rows.append({
                "folder": str(path.parent.relative_to(base_dir)),
                "filename": filename,
                "extension": path.suffix.lower(),
                "size_MB": round(path.stat().st_size / (1024 ** 2), 3),
                "full_path": str(path)
            })

    return pd.DataFrame(rows)


project_files = scan_directory(PROJECT_DIR)

print("Total files:", len(project_files))

display(project_files)

In [ ]:
file_summary = (
    project_files
    .groupby("extension")
    .agg(
        files=("filename", "count"),
        total_size_MB=("size_MB", "sum")
    )
    .sort_values("files", ascending=False)
)

display(file_summary)

In [ ]:
result_files = scan_directory(RESULTS_DIR)

experiment_summary = (
    result_files
    .groupby("folder")
    .agg(
        total_files=("filename", "count"),
        total_size_MB=("size_MB", "sum")
    )
    .sort_index()
)

display(experiment_summary)

In [ ]:
for folder in sorted(result_files["folder"].unique()):

    sub = result_files[
        result_files["folder"] == folder
    ]

    print("\n" + "=" * 90)
    print("📁", folder)
    print("Files:", len(sub))
    print("=" * 90)

    for row in sub.itertuples():
        print(
            f"{row.filename:<55} "
            f"{row.size_MB:>10.3f} MB"
        )

In [ ]:
dataset_files = scan_directory(DATASET_DIR)

dataset_summary = (
    dataset_files
    .groupby("folder")
    .agg(
        total_files=("filename", "count"),
        total_size_MB=("size_MB", "sum")
    )
    .sort_index()
)

display(dataset_summary)

In [ ]:
import pandas as pd

# เอาเฉพาะไฟล์ภาพใน folder ที่ลงท้ายด้วย /visualize
vis = project_files[project_files['folder'].str.contains(r'(^|/)visualize$', regex=True)]

# นับจำนวนภาพต่อโฟลเดอร์
summary = (vis.groupby('folder')
              .size()
              .reset_index(name='n_images')
              .sort_values('n_images'))

# เอาเฉพาะที่ไม่ครบ 23
incomplete = summary[summary['n_images'] != 23]

pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
print(f"โฟลเดอร์ visualize ทั้งหมด: {len(summary)}")
print(f"ไม่ครบ 23: {len(incomplete)}")
print(incomplete.to_string(index=False))

In [ ]:
import pandas as pd
import re

# ใช้ project_files ที่มีอยู่แล้ว
df = project_files.copy()

# 1) หาทุกโฟลเดอร์โมเดล (โฟลเดอร์ระดับบนสุดใต้ results/ ที่ขึ้นต้นด้วย CNN)
def model_root(folder):
    parts = folder.split('/')
    if parts[0] == 'results' and len(parts) > 1 and parts[1].startswith('CNN'):
        return '/'.join(parts[:2])
    return None

df['model'] = df['folder'].apply(model_root)
models = sorted(df['model'].dropna().unique())

# 2) หาภาพใน /visualize ของแต่ละโมเดล (รวมกรณีซ้อนลึก เช่น test_results/visualize)
vis = df[df['folder'].str.contains(r'(^|/)visualize$', regex=True)
         & df['filename'].str.match(r'test_SN\d+\.png')].copy()
vis['model'] = vis['folder'].apply(model_root)
vis['vis_folder'] = vis['folder']
vis['subject'] = vis['filename'].str.extract(r'(SN\d+)')

# 3) subject ทั้ง 23 คน: ดึงจาก hypnogram_full ของโมเดลที่ครบ
all_subjects = sorted(
    df[df['filename'].str.match(r'hypnogram_full_SN\d+\.png')]['filename']
      .str.extract(r'(SN\d+)')[0].unique()
)
print("จำนวน subject ทั้งหมด:", len(all_subjects))

# 4) สรุปรายโมเดล
rows = []
for m in models:
    v = vis[vis['model'] == m]
    n = v['subject'].nunique()
    rows.append({
        'model': m,
        'has_visualize': len(v) > 0,
        'vis_folder': v['vis_folder'].iloc[0] if len(v) else '-',
        'n_images': n,
        'missing_count': len(all_subjects) - n,
    })

report = pd.DataFrame(rows)

# 5) โฟลเดอร์ที่ "ไม่มี /visualize เลย" หรือ "มีไม่ครบ 23"
need_fix = report[report['n_images'] < len(all_subjects)].copy()

# กรองตามกลุ่มที่สนใจ (แก้ pattern ได้)
pat = r'(newDT|baseline|exp5|Exp8|Exp13|Exp23)'
need_fix_focus = need_fix[need_fix['model'].str.contains(pat, regex=True)]

pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 250)

print("\n=== ไม่ครบ 23 (เฉพาะกลุ่มที่สนใจ) ===")
print(need_fix_focus.sort_values('n_images').to_string(index=False))

print("\n=== ไม่มี /visualize เลย ===")
print(report[~report['has_visualize']]['model'].to_string(index=False))

In [ ]:
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")           # ไม่ต้อง show ตอนวนหลายโฟลเดอร์
import matplotlib.pyplot as plt
from pathlib import Path

from pathlib import Path
ROOT = Path("/content/drive/MyDrive/SleepStageClassificationProject/results")
print(ROOT, ROOT.exists())   # ควรขึ้น True

# ---- config (แก้ให้ตรงกับของเดิม) ----
FS = 100
EPOCH_LEN = 30 * FS
STAGES = ["W", "N1", "N2", "N3", "R"]
LABELS = STAGES
CH_NAMES = {4: ["ch1", "ch2", "ch3", "ch4"],          # แก้เป็นชื่อช่องจริง
            7: ["ch1", "ch2", "ch3", "ch4", "ch5", "ch6", "ch7"]}

# X_test กลางของแต่ละชนิดช่อง (ชี้ไปโฟลเดอร์ที่มีไฟล์จริง)
X_TEST_PATH = {
    7: ROOT / "CNN_5_7ch_newDT_seed42" / "X_test.npy",
    4: ROOT / "CNN_5_4ch_newDT_seed42" / "X_test.npy",
}
_X_cache = {}
def load_X(n_ch):
    if n_ch not in _X_cache:
        _X_cache[n_ch] = np.load(X_TEST_PATH[n_ch], mmap_mode="r")
    return _X_cache[n_ch]

def get_n_channels(folder):
    return 4 if "4ch" in folder.name else 7

def find_vis_dir(folder):
    # Exp8 อยู่ใต้ test_results, Exp23 อยู่ใต้ visual_Exp23_soft
    for sub in ["test_results", "visual_Exp23_soft", ""]:
        if (folder / sub / "test_predictions.csv").exists() or (folder / sub / "test_prediction_table.csv").exists():
            return folder / sub / "visualize"
    return folder / "visualize"

def load_table(folder):
    base = find_vis_dir(folder).parent
    for name in ["test_predictions.csv", "test_prediction_table.csv"]:
        p = base / name
        if p.exists():
            return pd.read_csv(p), p
    raise FileNotFoundError(f"ไม่เจอ test predictions ใน {base}")

def visualize_subject(table, X, subject, n_ch, out_dir, tag):
    ch_names = CH_NAMES[n_ch]
    idx = []
    for label in LABELS:
        rows = table.index[(table["subject"] == subject) & (table["actual"] == label)].to_numpy()
        if len(rows) == 0:
            return False
        idx.append(rows[0])
    t = np.arange(EPOCH_LEN) / FS
    fig, axes = plt.subplots(n_ch + 1, len(idx), figsize=(3.3*len(idx), 2*(n_ch+1)), squeeze=False)
    for col, i in enumerate(idx):
        row = table.loc[i]
        sig = np.asarray(X[i])[-EPOCH_LEN:, :]
        color = "green" if row["correct"] else "red"
        axes[0, col].set_title(f"Expert: {row['actual']}\nModel: {row['predicted']}", color=color, fontweight="bold")
        for c in range(n_ch):
            axes[c, col].plot(t, sig[:, c], lw=0.6)
            axes[c, col].set_xlim(0, 30); axes[c, col].grid(alpha=0.3)
        probs = [row["prob_" + s] for s in STAGES]
        axes[-1, col].bar(STAGES, probs, color="lightgray")
        axes[-1, col].bar(row["predicted"], row["prob_" + row["predicted"]], color=color)
        axes[-1, col].set_ylim(0, 1); axes[-1, col].set_xlabel("Probability")
    for c in range(n_ch):
        lo = min(axes[c, k].get_ylim()[0] for k in range(len(idx)))
        hi = max(axes[c, k].get_ylim()[1] for k in range(len(idx)))
        for k in range(len(idx)):
            axes[c, k].set_ylim(lo, hi)
        axes[c, 0].set_ylabel(ch_names[c])
    plt.suptitle(f"test | {subject} | raw signal (microvolt) of the last 30 s epoch, first sequence of each stage")
    plt.tight_layout()
    out_dir.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_dir / f"test_{subject}.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    return True

def run_folder(folder_name, overwrite=False):
    folder = ROOT / folder_name
    n_ch = get_n_channels(folder)
    table, path = load_table(folder)
    X = load_X(n_ch)
    if len(table) != len(X):
        print(f"[SKIP] {folder_name}: rows {len(table)} != X_test {len(X)} (index ไม่ตรง)")
        return
    out_dir = find_vis_dir(folder)
    done = 0
    for subject in sorted(table["subject"].unique()):
        if not overwrite and (out_dir / f"test_{subject}.png").exists():
            continue
        done += visualize_subject(table, X, subject, n_ch, out_dir, folder_name)
    print(f"[OK] {folder_name}: สร้างใหม่ {done} ภาพ -> {out_dir}")

TARGETS = [
    "CNN_5_7ch_newDT_seed43", "CNN_5_7ch_newDT_seed44",
    "CNN_LSTM_5_4ch_newDT_seed42", "CNN_LSTM_5_4ch_newDT_seed44",
    "CNN_LSTM_5_7ch_newDT_seed43", "CNN_LSTM_5_7ch_newDT_seed44",
    "CNN_5_4ch_newDT_seed42", "CNN_5_4ch_newDT_seed43",
    "CNN_5_7ch_newDT_seed42", "CNN_LSTM_5_4ch_newDT_seed43",
    "CNN_5_7ch_baseline_w_newdata", "CNN_LSTM_5_7ch_baseline_w_newdata",
    "CNN_5_7ch_exp5_seed42", "CNN_5_7ch_exp5_seed43", "CNN_5_7ch_exp5_seed44",
    "CNN_5_7ch_Exp8_class_weighted_seed42", "CNN_5_7ch_Exp8_class_weighted_seed43",
    "CNN_5_7ch_Exp8_class_weighted_seed44",
    "CNN_5_7ch_Exp13_test_ensemble", "CNN_5_7ch_Exp13_vs_Exp23_test",
]

for name in TARGETS:
    try:
        run_folder(name)
    except Exception as e:
        print(f"[ERR] {name}: {e}")

In [ ]:
for n in ["CNN_5_7ch_newDT_seed43", "CNN_5_7ch_exp5_seed42",
          "CNN_5_7ch_Exp8_class_weighted_seed43", "CNN_5_7ch_Exp13_test_ensemble"]:
    f = ROOT / n
    p = next(iter(sorted(f.rglob("test_predictions.csv"), key=lambda x: len(x.parts))), None)
    x = next(iter(f.rglob("X_test.npy")), None)
    print("\n==", n)
    print("pred:", p)
    print("X   :", x)
    if p:
        df = pd.read_csv(p)
        print("rows:", len(df), "| cols:", list(df.columns))
        print(df.head(3).to_string())
    if x:
        print("X shape:", np.load(x, mmap_mode="r").shape)

In [ ]:
import tensorflow as tf

def verify_folder(folder_name, n_check=300, seed=0):
    folder = ROOT / folder_name
    table_path = find_table_path(folder)
    xp = next(iter(sorted(folder.rglob("X_test.npy"), key=lambda p: len(p.parts))), None)
    mp = next(iter(sorted(list(folder.rglob("*best_model.keras")), key=lambda p: len(p.parts))), None)
    if not (table_path and xp and mp):
        print(f"[NO] {folder_name}: table={bool(table_path)} X={bool(xp)} model={bool(mp)}")
        return None
    table = pd.read_csv(table_path)
    X = np.load(xp, mmap_mode="r")
    if len(table) != len(X):
        print(f"[NO] {folder_name}: rows {len(table)} != X {len(X)}")
        return None
    rng = np.random.default_rng(seed)
    idx = np.sort(rng.choice(len(X), size=min(n_check, len(X)), replace=False))
    model = tf.keras.models.load_model(mp, compile=False)
    prob = model.predict(np.asarray(X[idx]), batch_size=32, verbose=0)
    pred = np.array(STAGES)[prob.argmax(1)]
    match = (pred == table.loc[idx, "predicted"].to_numpy()).mean()
    print(f"[{'PASS' if match > 0.95 else 'FAIL'}] {folder_name}: ตรงกัน {match:.1%} (n={len(idx)})")
    return match

verify_folder("CNN_5_7ch_newDT_seed43")

In [ ]:
import pandas as pd, numpy as np

BASELINE = [f"CNN{l}_5_{c}ch_{d}seed{s}" for l in ["", "_LSTM"] for c in [4, 7]
            for d in ["", "newDT_"] for s in [42, 43, 44]]
EXTRA = [f"CNN_5_7ch_exp5_seed{s}" for s in [42, 43, 44]] + \
        [f"CNN_5_7ch_Exp8_class_weighted_seed{s}" for s in [42, 43, 44]] + \
        ["CNN_5_7ch_Exp13_test_ensemble", "CNN_5_7ch_Exp13_vs_Exp23_test"]

rows = []
for n in BASELINE + EXTRA:
    f = ROOT / n
    if not f.exists():
        rows.append(dict(folder=n, status="NO_FOLDER")); continue
    tp = find_table_path(f)
    xp = next(iter(sorted(f.rglob("X_test.npy"), key=lambda p: len(p.parts))), None)
    mp = next(iter(sorted(f.rglob("*best_model.keras"), key=lambda p: len(p.parts))), None)
    nrow = nx = None; cols_ok = False
    if tp is not None:
        t = pd.read_csv(tp, nrows=None); nrow = len(t)
        cols_ok = {"subject", "actual", "predicted", "prob_W"}.issubset(t.columns)
    if xp is not None: nx = np.load(xp, mmap_mode="r").shape[0]
    vis = (tp.parent if tp else f) / "visualize"
    n_vis = len(list(vis.glob("test_SN*.png"))) if vis.exists() else 0
    rows.append(dict(folder=n, table=tp is not None, cols_ok=cols_ok, X=xp is not None,
                     model=mp is not None, rows=nrow, X_rows=nx,
                     rows_match=(nrow == nx) if nrow and nx else False, n_vis=n_vis))
report = pd.DataFrame(rows)
pd.set_option("display.max_rows", None); pd.set_option("display.width", 250)
print(report.to_string(index=False))

In [ ]:
print(ROOT.exists())
print([p.name for p in ROOT.iterdir()][:10])
f = ROOT / "CNN_5_7ch_newDT_seed42"
print(f.exists(), [p.name for p in f.iterdir()][:30] if f.exists() else "ไม่มีโฟลเดอร์")

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')   # ถ้า mount อยู่แล้วจะไม่เป็นไร

# 1) ดูว่าใน MyDrive มีอะไรบ้าง
print(os.listdir('/content/drive/MyDrive'))

# 2) ถ้า project_files ยังอยู่ในเซสชัน ดึง path เต็มจากมันเลย (แม่นสุด)
p = project_files[project_files['filename'] == 'test_hypnogram_3seeds.png']['full_path'].iloc[0]
print(p)

In [ ]:
import os, numpy as np, pandas as pd
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/SleepStageClassificationProject/results")
BASELINE = [f"CNN{l}_5_{c}ch_{d}seed{s}" for l in ["", "_LSTM"] for c in [4, 7]
            for d in ["", "newDT_"] for s in [42, 43, 44]]
EXTRA = [f"CNN_5_7ch_exp5_seed{s}" for s in [42, 43, 44]] + \
        [f"CNN_5_7ch_Exp8_class_weighted_seed{s}" for s in [42, 43, 44]] + \
        ["CNN_5_7ch_Exp13_test_ensemble", "CNN_5_7ch_Exp13_vs_Exp23_test",
         "CNN_5_7ch_Exp13_seed42", "CNN_5_7ch_Exp13_Seed43", "CNN_5_7ch_Exp13_Seed44",
         "CNN_5_7ch_Exp23_seed42", "CNN_5_7ch_Exp23_seed43", "CNN_5_7ch_Exp23_seed44"]
SCOPE = BASELINE + EXTRA

# ---------- 1) ตำแหน่งไฟล์สำคัญทุกไฟล์ ----------
SKIP_DIRS = {"hypnogram", "signal_sequences", "signal_whole_recording", "signal_window_30min",
             "temp_train", "temp_val", "temp_test"}
KEYS = ("test_pred", "test_sequences", "X_test", ".keras", "test_prediction_table",
        "test_experiment_row", "val_predictions")
rows = []
for name in SCOPE:
    f = ROOT / name
    if not f.exists():
        rows.append(dict(folder=name, file="(NO FOLDER)", rel_path="", size_MB=None)); continue
    for dp, dn, fn in os.walk(f):
        dn[:] = [d for d in dn if d not in SKIP_DIRS]
        p = Path(dp)
        if p.name == "visualize":
            n = len(list(p.glob("test_SN*.png")))
            rows.append(dict(folder=name, file=f"[visualize: {n} ภาพ]", rel_path=str(p.relative_to(f)), size_MB=None))
            continue
        for x in fn:
            if any(k in x for k in KEYS):
                fp = p / x
                rows.append(dict(folder=name, file=x, rel_path=str(fp.relative_to(f)),
                                 size_MB=round(fp.stat().st_size / 1e6, 3)))
inv = pd.DataFrame(rows)
pd.set_option("display.max_rows", None); pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", None)
print(inv.to_string(index=False))
inv.to_csv(ROOT.parent / "inventory_test_files.csv", index=False)

KEY = ["subject", "epoch_start", "epoch_end"]

# ---------- 2) กลุ่ม B: Exp8 seed43/44 เทียบกับ seed42 ----------
print("\n===== B: Exp8 =====")
ref_dir = ROOT / "CNN_5_7ch_Exp8_class_weighted_seed42" / "test_results"
ref_meta = pd.read_csv(ref_dir / "test_sequences.csv")
print("ref X_test.npy:", (ref_dir / "X_test.npy").exists(), "| meta rows:", len(ref_meta))
for s in [42, 43, 44]:
    d = ROOT / f"CNN_5_7ch_Exp8_class_weighted_seed{s}" / "test_results"
    print(f"\nseed{s} ไฟล์ใน test_results:", sorted(p.name for p in d.iterdir() if p.is_file()))
    t = pd.read_csv(d / "test_predictions.csv")
    same_key = len(t) == len(ref_meta) and (t[KEY].to_numpy() == ref_meta[KEY].to_numpy()).all()
    same_lab = len(t) == len(ref_meta) and (t["actual"].to_numpy() == ref_meta["stage"].to_numpy()).all()
    print(f"  แถว/คีย์ (subject,epoch) ตรงกับ meta ของ seed42: {same_key} | label ตรง: {same_lab}")
    own = d / "test_sequences.csv"
    if own.exists():
        print("  test_sequences.csv ของตัวเองตรงกับ seed42:", pd.read_csv(own).equals(ref_meta))

# ---------- 3) กลุ่ม D: Exp13 / Exp23 ----------
print("\n===== D: Exp13_test_ensemble =====")
d13 = ROOT / "CNN_5_7ch_Exp13_test_ensemble"
print("ไฟล์:", sorted(p.name for p in d13.iterdir() if p.is_file()))
t13 = pd.read_csv(d13 / "test_predictions.csv"); m13 = pd.read_csv(d13 / "test_sequences.csv")
print("rows pred/meta:", len(t13), len(m13), "| y_true == label ใน meta:", (t13["y_true"].to_numpy() == m13["label"].to_numpy()).all())
print("meta ตรงกับ meta ของ Exp8:", m13.equals(ref_meta))

print("\n===== D: Exp13_vs_Exp23_test =====")
d23 = ROOT / "CNN_5_7ch_Exp13_vs_Exp23_test"
print("ไฟล์:", sorted(p.name for p in d23.iterdir() if p.is_file()))
STG = ["W", "N1", "N2", "N3", "R"]
P = {}
for g in ["Exp13", "Exp23"]:
    for s in [42, 43, 44]:
        t = pd.read_csv(d23 / f"{g}_s{s}_test_predictions.csv")
        ok = (t[KEY].to_numpy() == ref_meta[KEY].to_numpy()).all()
        P[f"{g}_s{s}"] = t[["prob_" + k for k in STG]].to_numpy()
        print(f"{g}_s{s}: rows={len(t)} คีย์ตรง meta={ok}")
soft23 = np.mean([P[f"Exp23_s{s}"] for s in [42, 43, 44]], axis=0).argmax(1)
tab = pd.read_csv(d23 / "visual_Exp23_soft" / "test_prediction_table.csv")
print("soft vote Exp23 (คำนวณใหม่) ตรงกับ test_prediction_table.csv:",
      (np.array(STG)[soft23] == tab["predicted"].to_numpy()).mean())
print("คอลัมน์ของ test_prediction_table.csv:", list(tab.columns))

# ---------- 4) กลุ่ม C: ตรวจว่ามีโมเดลให้รัน test ไหม ----------
print("\n===== C =====")
C = ["CNN_5_4ch_newDT_seed44", "CNN_LSTM_5_4ch_newDT_seed42",
     "CNN_LSTM_5_4ch_newDT_seed44", "CNN_LSTM_5_7ch_newDT_seed43"]
for n in C:
    f = ROOT / n
    models = list(f.rglob("*.keras")) if f.exists() else []
    print(n, "| มีโฟลเดอร์:", f.exists(), "| โมเดล:", [m.name for m in models],
          "| ไฟล์ในโฟลเดอร์:", sorted(p.name for p in f.iterdir())[:8] if f.exists() else [])

In [ ]:
import gc, numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tensorflow as tf
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/SleepStageClassificationProject/results")
STAGES = ["W", "N1", "N2", "N3", "R"]
FS, EPOCH_LEN, THRESH = 100, 3000, 0.95
CH = {7: ["F4-M1", "C4-M1", "O2-M1", "C3-M2", "E1-M2", "E2-M2", "Chin"],
      4: ["ch1", "ch2", "ch3", "ch4"]}   # <-- แก้เป็นชื่อช่อง 4ch จริง

BASELINE = [f"CNN{l}_5_{c}ch_{d}seed{s}" for l in ["", "_LSTM"] for c in [4, 7]
            for d in ["", "newDT_"] for s in [42, 43, 44]]
SCOPE = BASELINE + [f"CNN_5_7ch_exp5_seed{s}" for s in [42, 43, 44]] + \
        [f"CNN_5_7ch_Exp8_class_weighted_seed{s}" for s in [42, 43, 44]]

def first(folder, pat):
    h = sorted(folder.rglob(pat), key=lambda p: len(p.parts))
    return h[0] if h else None

TODO, SUMMARY = [], []

def draw(table, X, out_dir, n_ch):
    table = table.reset_index(drop=True)
    out_dir.mkdir(parents=True, exist_ok=True)
    t = np.arange(EPOCH_LEN) / FS
    made = 0
    for s in sorted(table["subject"].unique()):
        f = out_dir / f"test_{s}.png"
        if f.exists():
            continue
        idx = []
        for st in STAGES:
            r = np.where((table["subject"] == s) & (table["actual"] == st))[0]
            if len(r) == 0: break
            idx.append(r[0])
        if len(idx) < 5:
            TODO.append((str(out_dir), f"{s}: ไม่มีครบ 5 stage ข้าม")); continue
        fig, axes = plt.subplots(n_ch + 1, 5, figsize=(16.5, 2 * (n_ch + 1)), squeeze=False)
        for col, i in enumerate(idx):
            row = table.loc[i]
            sig = np.asarray(X[i])[-EPOCH_LEN:, :]
            color = "green" if row["correct"] else "red"
            axes[0, col].set_title(f"Expert: {row['actual']}\nModel: {row['predicted']}", color=color, fontweight="bold")
            for c in range(n_ch):
                axes[c, col].plot(t, sig[:, c], lw=0.6); axes[c, col].set_xlim(0, 30); axes[c, col].grid(alpha=0.3)
            axes[-1, col].bar(STAGES, [row["prob_" + k] for k in STAGES], color="lightgray")
            axes[-1, col].bar(row["predicted"], row["prob_" + row["predicted"]], color=color)
            axes[-1, col].set_ylim(0, 1); axes[-1, col].set_xlabel("Probability")
        for c in range(n_ch):
            lo = min(axes[c, k].get_ylim()[0] for k in range(5)); hi = max(axes[c, k].get_ylim()[1] for k in range(5))
            for k in range(5): axes[c, k].set_ylim(lo, hi)
            axes[c, 0].set_ylabel(CH[n_ch][c])
        plt.suptitle(f"test | {s} | raw signal (microvolt) of the last 30 s epoch, first sequence of each stage")
        plt.tight_layout()
        plt.savefig(f, dpi=150, bbox_inches="tight"); plt.close(fig); made += 1
    return made

def verify(model_path, X, table, n=300):
    idx = np.sort(np.random.default_rng(0).choice(len(X), min(n, len(X)), replace=False))
    m = tf.keras.models.load_model(model_path, compile=False)
    pred = np.array(STAGES)[m.predict(np.asarray(X[idx]), batch_size=32, verbose=0).argmax(1)]
    del m; tf.keras.backend.clear_session(); gc.collect()
    return float((pred == table["predicted"].to_numpy()[idx]).mean())

def finish(name, status, match, made, out_dir):
    n_vis = len(list(out_dir.glob("test_SN*.png"))) if out_dir.exists() else 0
    SUMMARY.append(dict(folder=name, status=status, match=match, drawn_now=made, n_vis=n_vis))
    print(f"[{status}] {name} match={match} วาดใหม่={made} รวม={n_vis}")

# ---------- กลุ่มโมเดลเดี่ยว (baseline / exp5 / Exp8) ----------
EXP8_X = ROOT / "CNN_5_7ch_Exp8_class_weighted_seed42" / "test_results" / "X_test.npy"
for name in SCOPE:
    try:
        f = ROOT / name
        tp = first(f, "test_predictions.csv"); mp = first(f, "*best_model.keras")
        xp = first(f, "X_test.npy")
        if "Exp8_class_weighted_seed4" in name and xp is None:
            xp = EXP8_X          # ยืมสัญญาณดิบชุด test เดียวกัน (คีย์ตรง 100%)
        if tp is None: TODO.append((name, "ไม่มี test_predictions.csv (ต้องรัน test)")); continue
        if mp is None: TODO.append((name, "ไม่มีโมเดล .keras (ต้องเทรน/หาโมเดล)")); continue
        if xp is None: TODO.append((name, "ไม่มี X_test.npy ของรุ่นนี้")); continue
        table = pd.read_csv(tp); X = np.load(xp, mmap_mode="r")
        if len(table) != len(X):
            TODO.append((name, f"แถว {len(table)} != X {len(X)} (คนละชุด test)")); continue
        match = verify(mp, X, table)
        out_dir = tp.parent / "visualize"
        if match < THRESH:
            TODO.append((name, f"verify FAIL ({match:.1%}) ไม่วาด")); finish(name, "FAIL", match, 0, out_dir); continue
        made = draw(table, X, out_dir, X.shape[2])
        finish(name, "PASS", round(match, 4), made, out_dir)
    except Exception as e:
        TODO.append((name, f"ERROR {e}")); print("[ERR]", name, e)

# ---------- Exp23 soft (เฉลี่ย prob 3 seed) และ Exp13 (สัดส่วนโหวต) ----------
try:
    d23 = ROOT / "CNN_5_7ch_Exp13_vs_Exp23_test"; d13 = ROOT / "CNN_5_7ch_Exp13_test_ensemble"
    meta = pd.read_csv(d13 / "test_sequences.csv")
    base = meta[["subject", "epoch_start", "epoch_end"]].copy()
    base["actual"] = meta["stage"].to_numpy()

    # Exp23
    P = np.mean([pd.read_csv(d23 / f"Exp23_s{s}_test_predictions.csv")[["prob_" + k for k in STAGES]].to_numpy()
                 for s in [42, 43, 44]], axis=0)
    t23 = base.copy()
    t23["predicted"] = np.array(STAGES)[P.argmax(1)]
    for i, k in enumerate(STAGES): t23["prob_" + k] = P[:, i]
    t23["correct"] = t23["actual"] == t23["predicted"]
    ref = pd.read_csv(d23 / "visual_Exp23_soft" / "test_prediction_table.csv")
    m23 = float((ref["predicted"].to_numpy() == t23["predicted"].to_numpy()).mean())
    out23 = d23 / "visual_Exp23_soft" / "visualize"
    if m23 > 0.999:
        finish("Exp23_soft", "PASS", m23, draw(t23, np.load(d23 / "X_test.npy", mmap_mode="r"), out23, 7), out23)
    else:
        TODO.append(("Exp23_soft", f"ไม่ตรงตารางเดิม {m23:.1%}"))

    # Exp13 (โหวตแข็ง, เสมอ -> seed42)
    V = np.stack([pd.read_csv(d23 / f"Exp13_s{s}_test_predictions.csv")["predicted"].map(STAGES.index).to_numpy()
                  for s in [42, 43, 44]])
    cnt = np.stack([(V == k).sum(0) for k in range(5)], axis=1)
    tie = (cnt == cnt.max(1, keepdims=True)).sum(1) > 1
    vote = cnt.argmax(1); vote[tie] = V[0][tie]
    old = pd.read_csv(d13 / "test_predictions.csv")["pred_ensemble"].to_numpy()
    m13 = float((old == vote).mean())
    t13 = base.copy()
    t13["predicted"] = np.array(STAGES)[old]
    for i, k in enumerate(STAGES): t13["prob_" + k] = cnt[:, i] / 3
    t13["correct"] = t13["actual"] == t13["predicted"]
    out13 = d13 / "visualize"
    if m13 > 0.999:
        finish("Exp13_vote", "PASS", m13, draw(t13, np.load(d13 / "X_test.npy", mmap_mode="r"), out13, 7), out13)
    else:
        TODO.append(("Exp13_vote", f"โหวตคำนวณใหม่ไม่ตรงของเดิม {m13:.1%}"))
except Exception as e:
    TODO.append(("Exp13/Exp23", f"ERROR {e}")); print("[ERR] Exp13/23", e)

pd.DataFrame(SUMMARY).to_csv(ROOT.parent / "visualize_summary.csv", index=False)
pd.DataFrame(TODO, columns=["folder", "reason"]).to_csv(ROOT.parent / "TODO_missing.csv", index=False)
print("\n===== TODO (ยังไม่มี/ทำไม่ได้) =====")
print(pd.DataFrame(TODO, columns=["folder", "reason"]).to_string(index=False))

In [ ]:
import gc, re, numpy as np, pandas as pd, tensorflow as tf
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/SleepStageClassificationProject/results")
STAGES = ["W", "N1", "N2", "N3", "R"]
KEY = ["subject", "epoch_start", "epoch_end"]
N_VERIFY = 300

BASELINE = [f"CNN{l}_5_{c}ch_{d}seed{s}" for l in ["", "_LSTM"] for c in [4, 7]
            for d in ["", "newDT_"] for s in [42, 43, 44]]
SCOPE = BASELINE + [f"CNN_5_7ch_exp5_seed{s}" for s in [42, 43, 44]] + \
        [f"CNN_5_7ch_Exp8_class_weighted_seed{s}" for s in [42, 43, 44]]

def first(folder, pat):
    h = sorted(folder.rglob(pat), key=lambda p: len(p.parts)) if folder.exists() else []
    return h[0] if h else None

# แหล่ง X ชุด 3032 แถว + ตารางคีย์ (ใช้จับคู่ให้รุ่นเก่า)
REF = {
    7: (ROOT / "CNN_5_7ch_Exp13_test_ensemble" / "X_test.npy",
        ROOT / "CNN_5_7ch_Exp13_test_ensemble" / "test_sequences.csv", "stage"),
    4: (ROOT / "CNN_5_4ch_newDT_seed42" / "X_test.npy",
        ROOT / "CNN_5_4ch_newDT_seed42" / "test_predictions.csv", "actual"),
}
EXP8_X = ROOT / "CNN_5_7ch_Exp8_class_weighted_seed42" / "test_results" / "X_test.npy"
for n_ch, (xp, kp, _) in REF.items():
    print(f"REF {n_ch}ch: X={xp.exists()} keys={kp.exists()}")

rows = []
for name in SCOPE:
    r = dict(folder=name, plan="-", note="")
    try:
        f = ROOT / name
        n_ch = 4 if "4ch" in name else 7
        tp, mp = first(f, "test_predictions.csv"), first(f, "*best_model.keras")
        r["has_pred"], r["has_model"] = tp is not None, mp is not None
        if tp is None or mp is None:
            r["plan"] = "TODO"; r["note"] = "ไม่มี " + ("pred " if tp is None else "") + ("model" if mp is None else "")
            rows.append(r); continue

        t = pd.read_csv(tp)
        need = set(KEY + ["actual", "predicted", "correct"] + ["prob_" + s for s in STAGES])
        r["cols_ok"] = need.issubset(t.columns); r["rows"] = len(t)
        if not r["cols_ok"]:
            r["plan"] = "TODO"; r["note"] = f"คอลัมน์ขาด {sorted(need - set(t.columns))}"; rows.append(r); continue

        # --- เลือกแหล่ง X ---
        own = first(f, "X_test.npy")
        xref, kref, lab_col = REF[n_ch]
        if own is not None:
            xp, how = own, "own"
        elif "Exp8_class_weighted_seed4" in name:
            xp, how = EXP8_X, "borrow_exp8_seed42"
        else:
            xp, how = xref, "match_by_key"
        X = np.load(xp, mmap_mode="r")
        r["x_source"], r["X_rows"] = how, X.shape[0]

        # --- จัดแถว X ให้ตรงกับ CSV ---
        if how == "match_by_key" or (how != "own" and len(t) != len(X)) or (how == "borrow_exp8_seed42" and len(t) != len(X)):
            ref = pd.read_csv(kref)
            refk = ref[KEY].reset_index().rename(columns={"index": "ref_idx"})
            mg = t.merge(refk, on=KEY, how="left")
            r["keys_found"] = f"{int(mg['ref_idx'].notna().sum())}/{len(t)}"
            if mg["ref_idx"].isna().any():
                r["plan"] = "TODO"; r["note"] = "คีย์จับคู่ไม่ครบ (คนละชุด test)"; rows.append(r); continue
            lab_ok = (ref.loc[mg["ref_idx"].astype(int), lab_col].to_numpy() == t["actual"].to_numpy()).all()
            r["label_ok"] = bool(lab_ok)
            if not lab_ok:
                r["plan"] = "TODO"; r["note"] = "label ไม่ตรง"; rows.append(r); continue
            sel = mg["ref_idx"].astype(int).to_numpy()
        else:
            if len(t) != len(X):
                r["plan"] = "TODO"; r["note"] = f"แถว {len(t)} != X {len(X)}"; rows.append(r); continue
            sel = np.arange(len(t))
            if how == "borrow_exp8_seed42":
                ref = pd.read_csv(ROOT / "CNN_5_7ch_Exp8_class_weighted_seed42" / "test_results" / "test_sequences.csv")
                r["keys_found"] = f"{len(t)}/{len(t)}"
                r["label_ok"] = bool((t[KEY].to_numpy() == ref[KEY].to_numpy()).all()
                                     and (t["actual"].to_numpy() == ref["stage"].to_numpy()).all())
                if not r["label_ok"]:
                    r["plan"] = "TODO"; r["note"] = "คีย์/label ไม่ตรง Exp8 seed42"; rows.append(r); continue

        # --- verify ด้วยโมเดลตัวเอง ---
        pick = np.sort(np.random.default_rng(0).choice(len(t), min(N_VERIFY, len(t)), replace=False))
        Xs = np.stack([X[int(sel[i])] for i in pick])
        m = tf.keras.models.load_model(mp, compile=False)
        pred = np.array(STAGES)[m.predict(Xs, batch_size=32, verbose=0).argmax(1)]
        del m, Xs; tf.keras.backend.clear_session(); gc.collect()
        r["verify"] = round(float((pred == t["predicted"].to_numpy()[pick]).mean()), 4)

        # --- ตรวจโฟลเดอร์ visualize ---
        vis = tp.parent / "visualize"
        files = sorted(p.name for p in vis.glob("*.png")) if vis.exists() else []
        have = {m_.group(1) for x in files if (m_ := re.fullmatch(r"test_(SN\d+)\.png", x))}
        subs = set(t["subject"]); five = {s for s in subs if set(t.loc[t.subject == s, "actual"]) == set(STAGES)}
        r["n_vis"] = len(have); r["missing"] = len(five - have)
        extra = [x for x in files if not re.fullmatch(r"test_(SN\d+)\.png", x)] + sorted(have - subs)
        r["extra_files"] = len(extra)
        r["note"] = ("ไฟล์เกิน: " + ", ".join(extra[:6])) if extra else ""
        r["plan"] = "DRAW" if r["verify"] >= 0.95 else "TODO"
        if r["plan"] == "TODO": r["note"] += f" verify FAIL"
    except Exception as e:
        r["plan"] = "TODO"; r["note"] = f"ERROR {e}"
    rows.append(r)
    print(r["folder"], r["plan"], r.get("verify"), r["note"][:60])

rep = pd.DataFrame(rows)
cols = ["folder", "plan", "x_source", "rows", "keys_found", "label_ok", "verify", "n_vis", "missing", "extra_files", "note"]
rep = rep.reindex(columns=cols)
pd.set_option("display.max_rows", None); pd.set_option("display.width", 300); pd.set_option("display.max_colwidth", 70)
print(rep.to_string(index=False))
rep.to_csv(ROOT.parent / "precheck_report.csv", index=False)

# ชื่อช่อง 4ch: หาจากไฟล์ที่มีในโปรเจกต์
print("\n--- ค้นหาชื่อช่อง 4ch ---")
for p in list((ROOT.parent).rglob("*.py"))[:50] + list(ROOT.rglob("experiment_row.csv"))[:3]:
    try:
        txt = p.read_text(errors="ignore")
        for m_ in re.finditer(r"SELECTED_CHANNELS\s*=\s*\[[^\]]*\]", txt):
            print(p.name, "->", m_.group(0)[:150])
    except Exception:
        pass

In [ ]:
import gc, shutil, numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tensorflow as tf
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/SleepStageClassificationProject/results")
STAGES = ["W", "N1", "N2", "N3", "R"]
KEY = ["subject", "epoch_start", "epoch_end"]
FS, EPOCH_LEN = 100, 3000
CH4 = ["C4-M1", "C3-M2", "E1-M2", "Chin"]           # <-- แก้เป็นชื่อช่อง 4ch จริง
CH7 = ["F4-M1", "C4-M1", "O2-M1", "C3-M2", "E1-M2", "E2-M2", "Chin"]
REDRAW_OLD = False   # True = วาดรุ่นเก่า 115 แถวใหม่ทั้ง 23 (สำรองของเดิม), False = เติมเฉพาะที่ขาด

BASELINE = [f"CNN{l}_5_{c}ch_{d}seed{s}" for l in ["", "_LSTM"] for c in [4, 7]
            for d in ["", "newDT_"] for s in [42, 43, 44]]
SCOPE = [n for n in BASELINE if n != "CNN_LSTM_5_7ch_newDT_seed43"] + \
        [f"CNN_5_7ch_exp5_seed{s}" for s in [42, 43, 44]] + \
        [f"CNN_5_7ch_Exp8_class_weighted_seed{s}" for s in [42, 43, 44]]

def first(folder, pat):
    h = sorted(folder.rglob(pat), key=lambda p: len(p.parts)) if folder.exists() else []
    return h[0] if h else None

REF = {7: (ROOT/"CNN_5_7ch_Exp13_test_ensemble"/"X_test.npy", ROOT/"CNN_5_7ch_Exp13_test_ensemble"/"test_sequences.csv"),
       4: (ROOT/"CNN_5_4ch_newDT_seed42"/"X_test.npy", ROOT/"CNN_5_4ch_newDT_seed42"/"test_predictions.csv")}
EXP8_X = ROOT/"CNN_5_7ch_Exp8_class_weighted_seed42"/"test_results"/"X_test.npy"
TODO, SUMMARY = [("CNN_LSTM_5_7ch_newDT_seed43", "ไม่มีโมเดลและ test_predictions (ต้องเทรน/รัน test)")], []

def draw(table, getX, out_dir, names, overwrite=False):
    table = table.reset_index(drop=True); out_dir.mkdir(parents=True, exist_ok=True)
    n_ch = len(names); t = np.arange(EPOCH_LEN) / FS; made = 0
    for s in sorted(table["subject"].unique()):
        f = out_dir / f"test_{s}.png"
        if f.exists() and not overwrite: continue
        idx = []
        for st in STAGES:
            r = np.where((table["subject"] == s) & (table["actual"] == st))[0]
            if len(r) == 0: break
            idx.append(r[0])
        if len(idx) < 5: TODO.append((str(out_dir), f"{s}: ไม่มีครบ 5 stage")); continue
        fig, axes = plt.subplots(n_ch + 1, 5, figsize=(16.5, 2 * (n_ch + 1)), squeeze=False)
        for col, i in enumerate(idx):
            row = table.loc[i]; sig = np.asarray(getX(i))[-EPOCH_LEN:, :]
            color = "green" if row["correct"] else "red"
            axes[0, col].set_title(f"Expert: {row['actual']}\nModel: {row['predicted']}", color=color, fontweight="bold")
            for c in range(n_ch):
                axes[c, col].plot(t, sig[:, c], lw=0.6); axes[c, col].set_xlim(0, 30); axes[c, col].grid(alpha=0.3)
            axes[-1, col].bar(STAGES, [row["prob_" + k] for k in STAGES], color="lightgray")
            axes[-1, col].bar(row["predicted"], row["prob_" + row["predicted"]], color=color)
            axes[-1, col].set_ylim(0, 1); axes[-1, col].set_xlabel("Probability")
        for c in range(n_ch):
            lo = min(axes[c, k].get_ylim()[0] for k in range(5)); hi = max(axes[c, k].get_ylim()[1] for k in range(5))
            for k in range(5): axes[c, k].set_ylim(lo, hi)
            axes[c, 0].set_ylabel(names[c])
        plt.suptitle(f"test | {s} | raw signal (microvolt) of the last 30 s epoch, first sequence of each stage")
        plt.tight_layout(); plt.savefig(f, dpi=150, bbox_inches="tight"); plt.close(fig); made += 1
    return made

def done(name, made, out_dir, note=""):
    n = len([p for p in out_dir.glob("test_SN*.png") if " (" not in p.name])
    SUMMARY.append(dict(folder=name, drawn_now=made, n_vis=n, note=note)); print(name, "วาดใหม่", made, "รวม", n, note)

# ---- โมเดลเดี่ยว (ผ่าน verify = 1.0 แล้ว) ----
for name in SCOPE:
    try:
        f = ROOT / name; n_ch = 4 if "4ch" in name else 7; names = CH4 if n_ch == 4 else CH7
        tp = first(f, "test_predictions.csv"); t = pd.read_csv(tp)
        own = first(f, "X_test.npy"); old = len(t) == 115
        if own is not None and len(t) == len(np.load(own, mmap_mode="r")):
            X = np.load(own, mmap_mode="r"); getX = lambda i, X=X: X[i]
        elif "Exp8_class_weighted_seed4" in name:
            X = np.load(EXP8_X, mmap_mode="r"); getX = lambda i, X=X: X[i]
        else:
            xref, kref = REF[n_ch]; X = np.load(xref, mmap_mode="r")
            ref = pd.read_csv(kref)[KEY].reset_index().rename(columns={"index": "ref_idx"})
            sel = t.merge(ref, on=KEY, how="left")["ref_idx"].astype(int).to_numpy()
            getX = lambda i, X=X, sel=sel: X[sel[i]]
        out_dir = tp.parent / "visualize"; ow = False
        if old and REDRAW_OLD:
            if out_dir.exists() and not (tp.parent / "visualize_old_backup").exists():
                shutil.copytree(out_dir, tp.parent / "visualize_old_backup")
            ow = True
        done(name, draw(t, getX, out_dir, names, ow), out_dir)
    except Exception as e:
        TODO.append((name, f"ERROR {e}")); print("[ERR]", name, e)

# ---- Exp23 soft / Exp13 vote ----
try:
    d23 = ROOT/"CNN_5_7ch_Exp13_vs_Exp23_test"; d13 = ROOT/"CNN_5_7ch_Exp13_test_ensemble"
    meta = pd.read_csv(d13/"test_sequences.csv"); base = meta[KEY].copy(); base["actual"] = meta["stage"].to_numpy()
    P = np.mean([pd.read_csv(d23/f"Exp23_s{s}_test_predictions.csv")[["prob_"+k for k in STAGES]].to_numpy() for s in [42,43,44]], axis=0)
    t23 = base.copy(); t23["predicted"] = np.array(STAGES)[P.argmax(1)]
    for i, k in enumerate(STAGES): t23["prob_"+k] = P[:, i]
    t23["correct"] = t23["actual"] == t23["predicted"]
    ref = pd.read_csv(d23/"visual_Exp23_soft"/"test_prediction_table.csv")
    if (ref["predicted"].to_numpy() == t23["predicted"].to_numpy()).mean() > 0.999:
        X = np.load(d23/"X_test.npy", mmap_mode="r"); o = d23/"visual_Exp23_soft"/"visualize"
        done("Exp23_soft", draw(t23, lambda i: X[i], o, CH7), o)
    else: TODO.append(("Exp23_soft", "ไม่ตรงตารางเดิม"))

    V = np.stack([pd.read_csv(d23/f"Exp13_s{s}_test_predictions.csv")["predicted"].map(STAGES.index).to_numpy() for s in [42,43,44]])
    cnt = np.stack([(V == k).sum(0) for k in range(5)], axis=1)
    tie = (cnt == cnt.max(1, keepdims=True)).sum(1) > 1; vote = cnt.argmax(1); vote[tie] = V[0][tie]
    oldp = pd.read_csv(d13/"test_predictions.csv")["pred_ensemble"].to_numpy()
    if (oldp == vote).mean() > 0.999:
        t13 = base.copy(); t13["predicted"] = np.array(STAGES)[oldp]
        for i, k in enumerate(STAGES): t13["prob_"+k] = cnt[:, i] / 3
        t13["correct"] = t13["actual"] == t13["predicted"]
        X = np.load(d13/"X_test.npy", mmap_mode="r"); o = d13/"visualize"
        done("Exp13_vote", draw(t13, lambda i: X[i], o, CH7), o, "กราฟแท่ง = สัดส่วนโหวต 3 seed")
    else: TODO.append(("Exp13_vote", "โหวตไม่ตรงของเดิม"))
except Exception as e:
    TODO.append(("Exp13/Exp23", f"ERROR {e}")); print("[ERR]", e)

pd.DataFrame(SUMMARY).to_csv(ROOT.parent/"visualize_summary.csv", index=False)
pd.DataFrame(TODO, columns=["folder", "reason"]).to_csv(ROOT.parent/"TODO_missing.csv", index=False)
print(pd.DataFrame(SUMMARY).to_string(index=False)); print("\nTODO:"); print(pd.DataFrame(TODO, columns=["folder","reason"]).to_string(index=False))

In [ ]:
import re, numpy as np, pandas as pd
from pathlib import Path
from collections import Counter

ROOT = Path("/content/drive/MyDrive/SleepStageClassificationProject/results")
BASELINE = [f"CNN{l}_5_{c}ch_{d}seed{s}" for l in ["", "_LSTM"] for c in [4, 7]
            for d in ["", "newDT_"] for s in [42, 43, 44]]
IMG_DIRS = {"visualize": r"test_(SN\d+)\.png",
            "hypnogram": r"hypnogram_full_(SN\d+)\.png",
            "signal_sequences": r"signal_sequences_(SN\d+)(?:_p\d+)?\.png",
            "signal_whole_recording": r"signal_whole_(SN\d+)\.png"}

def group_of(n):
    return ("LSTM" if "LSTM" in n else "CNN", "4ch" if "4ch" in n else "7ch",
            "newDT" if "newDT" in n else "old")

# ---- 1) สำรวจไฟล์ระดับบนสุดของแต่ละโฟลเดอร์ + นับภาพ 4 ชุด ----
info = {}
for n in BASELINE:
    f = ROOT / n
    d = dict(exists=f.exists(), files=set(), subs={}, extra={}, csv_rows={}, empty=[])
    if f.exists():
        for p in f.iterdir():
            if p.is_file():
                d["files"].add(p.name)
                if p.stat().st_size == 0: d["empty"].append(p.name)
                if p.name in ("val_predictions.csv", "test_predictions.csv"):
                    d["csv_rows"][p.name] = len(pd.read_csv(p))
        for sub, pat in IMG_DIRS.items():
            sd = f / sub
            if not sd.exists(): d["subs"][sub] = None; continue
            names = [x.name for x in sd.glob("*.png")]
            ok = {m.group(1) for x in names if (m := re.fullmatch(pat, x))}
            d["subs"][sub] = ok
            d["extra"][sub] = [x for x in names if not re.fullmatch(pat, x)]
    info[n] = d

# ---- 2) ไฟล์แบบฉบับของแต่ละกลุ่ม = ไฟล์ที่ >=50% ของสมาชิกมี ----
grp = {}
for n in BASELINE:
    if info[n]["exists"]: grp.setdefault(group_of(n), []).append(n)
canon = {}
for g, ms in grp.items():
    c = Counter(x for m in ms for x in info[m]["files"])
    canon[g] = {x for x, k in c.items() if k >= len(ms) / 2}

# subject ทั้ง 23 คน: ใช้ union จากภาพ hypnogram ทุกโฟลเดอร์
ALL = set()
for n in BASELINE:
    for s in info[n]["subs"].values():
        if s: ALL |= s
ALL = {s for s in ALL if s}
print("subject ทั้งหมดที่พบ:", len(ALL))

rows = []
for n in BASELINE:
    d = info[n]; r = dict(folder=n)
    if not d["exists"]:
        r["problem"] = "ไม่มีโฟลเดอร์"; rows.append(r); continue
    miss_files = sorted(canon[group_of(n)] - d["files"])
    r["missing_files"] = ", ".join(miss_files)
    for sub in IMG_DIRS:
        s = d["subs"][sub]
        r[f"{sub}"] = "ไม่มีโฟลเดอร์" if s is None else f"{len(s)}/{len(ALL)}"
        r[f"{sub}_ขาด"] = "" if s is None else ",".join(sorted(ALL - s))
    r["extra_img"] = "; ".join(f"{k}:{len(v)}" for k, v in d["extra"].items() if v)
    r["empty_files"] = ", ".join(d["empty"])
    r["val_rows"] = d["csv_rows"].get("val_predictions.csv")
    r["test_rows"] = d["csv_rows"].get("test_predictions.csv")
    rows.append(r)

rep = pd.DataFrame(rows)
pd.set_option("display.max_rows", None); pd.set_option("display.width", 320); pd.set_option("display.max_colwidth", 90)

print("\n===== A) ไฟล์แบบฉบับของแต่ละกลุ่ม =====")
for g, c in canon.items(): print(g, "->", len(c), "ไฟล์")

print("\n===== B) ไฟล์ระดับบนที่ขาด (เทียบกับเพื่อนร่วมกลุ่ม) =====")
print(rep[["folder", "missing_files"]].to_string(index=False))

print("\n===== C) ภาพครบ 23 ไหม =====")
print(rep[["folder"] + list(IMG_DIRS)].to_string(index=False))

print("\n===== D) subject ที่ขาดในแต่ละชุดภาพ (เฉพาะที่ขาด) =====")
for _, r in rep.iterrows():
    for sub in IMG_DIRS:
        if r.get(f"{sub}_ขาด"): print(r["folder"], "|", sub, "| ขาด:", r[f"{sub}_ขาด"])

print("\n===== E) จำนวนแถว val/test + ไฟล์เกิน/ว่างเปล่า =====")
print(rep[["folder", "val_rows", "test_rows", "extra_img", "empty_files"]].to_string(index=False))

rep.to_csv(ROOT.parent / "baseline_completeness_report.csv", index=False)
print("\nบันทึกที่", ROOT.parent / "baseline_completeness_report.csv")